In [4]:
import torchvision
from azureml.core import Workspace, Dataset, Datastore
from azureml.contrib.dataset import Dataset

subscription_id = 'f375b912-331c-4fc5-8e9f-2d7205e3e036'
resource_group = 'LabelingDemoRG'
workspace_name = 'labeling_demo_1'

workspace = Workspace(subscription_id, resource_group, workspace_name)

labeled_dataset = Dataset.get_by_name(workspace, name='Domestic Animals Detection-2019-10-26 00:35:21')
labeled_dataset.label['type'] = 'ObjectDetection'
labeled_dataset.image['column'] = 'image_url'
pytorch_dataset = labeled_dataset._to_torchvision()

In [5]:
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
# from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

import torch
import utils
from azureml.core import Workspace, Dataset, Datastore
from azureml.contrib.dataset import Dataset
from engine import train_one_epoch, evaluate

def get_model_instance_segmentation(num_classes):
    # load an instance segmentation model pre-trained pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def foo(): 
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    num_classes = 4

    indices = torch.randperm(len(pytorch_dataset)).tolist()
    dataset = torch.utils.data.Subset(pytorch_dataset, indices[:126])
    dataset_test = torch.utils.data.Subset(pytorch_dataset, indices[-20:])
    data_loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=utils.collate_fn)
    length = len(data_loader)
    data_loader_test = torch.utils.data.DataLoader(dataset_test, batch_size=2, shuffle=False, num_workers=0, collate_fn=utils.collate_fn)
    model = get_model_instance_segmentation(num_classes)
    model.to(device)


    # construct an optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    # and a learning rate scheduler
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    num_epochs = 10
    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
        # update the learning rate
        lr_scheduler.step()
        # evaluate on the test dataset
        evaluate(model, data_loader_test, device=device)
    pass


if __name__ == '__main__':
    foo()

Epoch: [0]  [ 0/63]  eta: 2:07:44  lr: 0.000086  loss: 1.9583 (1.9583)  loss_classifier: 1.7393 (1.7393)  loss_box_reg: 0.1857 (0.1857)  loss_objectness: 0.0018 (0.0018)  loss_rpn_box_reg: 0.0314 (0.0314)  time: 121.6590  data: 27.6674
Epoch: [0]  [10/63]  eta: 1:52:35  lr: 0.000891  loss: 1.2433 (1.1711)  loss_classifier: 1.1164 (1.0042)  loss_box_reg: 0.1347 (0.1343)  loss_objectness: 0.0018 (0.0054)  loss_rpn_box_reg: 0.0219 (0.0273)  time: 127.4636  data: 16.3855


KeyboardInterrupt: 